# Episode 12: Distributed training in PyTorch: 4 GPUs, DDP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep012-distributed.ipynb)

We pretrain a model about six times bigger, in GPT-2 small's shape, on four RTX 4090s rented by the hour, with PyTorch's DistributedDataParallel, and measure how the speed scales from one GPU to four, and how much better the model gets in thirty minutes.

[The video](https://youtu.be/5-RJVvuGZyE) · [The episode's page](https://mvgpt.nenuadrian.com/ep012-distributed/) · [The docs](https://nenuadrian.github.io/mvGPT/distributed/) · [The code at `ep012/end`](https://github.com/nenuadrian/mvGPT/tree/ep012/end)

This notebook runs the episode's commands on its code, as the episode left it. Models and runs from the episodes come from [the Hub](https://huggingface.co/nenuadrian/mvGPT-checkpoints), the files the videos used. What you train here comes out a little different from the video's, as training runs for a set time, on another machine. It runs on a GPU runtime, and Colab's free T4 will do.

In [ ]:
# The code as the episode left it, at its last tag, and what it needs.
import os

if not os.path.isdir("/content/mvGPT"):
    !git -c advice.detachedHead=false clone --quiet --depth 1 --branch ep012/end https://github.com/nenuadrian/mvGPT /content/mvGPT
%cd /content/mvGPT
!pip install --quiet -r requirements.txt
!python setup.py -q build_ext --inplace  # the C++ BPE tokenizer, as a Python module

import torch

print(torch.cuda.get_device_name() if torch.cuda.is_available() else "No GPU: Runtime > Change runtime type > T4 GPU")

from huggingface_hub import hf_hub_download


def fetch(*names):
    # Models from the episodes, from the Hub: the files the videos used, where the scripts look for them.
    for name in names:
        print(hf_hub_download("nenuadrian/mvGPT-checkpoints", f"checkpoints/{name}/model.pt", local_dir="."))


fetch("pretrain-rope-bpe", "pretrain-ddp")

## The data

In [ ]:
!python data.py --bpe  # TinyStories, the BPE merges, and the stories as BPE token ids

## Trying it on the CPU
Two processes, each with its own copy of a tiny model, averaging their gradients over gloo, as
on the laptop. The GPU is hidden from them, as two processes can't share one with NCCL.

In [ ]:
!CUDA_VISIBLE_DEVICES= torchrun --nnodes 1 --nproc_per_node 2 --master_addr 127.0.0.1 pretrain_ddp.py 0.25 --layers 2 --heads 2 --embd 64 --block 64 --batch 8 --name ddp-try 2>/dev/null

## On the GPU
The episode rented four RTX 4090s and trained the 91-million-parameter model for two minutes
on 1, 2 and 4 of them, then for 30 minutes on all four, on all of TinyStories
(`python data.py --full`, then `--full`). A Colab runtime has one GPU, so here is the
two-minute run on it, on the stories we have. With more GPUs, `--nproc_per_node` is how many.

In [ ]:
!nvidia-smi

In [ ]:
import torch

# bfloat16 in hardware: an L4 or A100 runtime has it, the free T4 (compute capability 7.5) doesn't.
AMPERE = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
print("bfloat16 in hardware" if AMPERE else "no bfloat16 in hardware: these runs will be slow")

In [ ]:
# torch.compile is left out without bfloat16 in hardware.
COMPILE = "--compile" if AMPERE else ""
!torchrun --standalone --nproc_per_node 1 pretrain_ddp.py 2 {COMPILE} --name scale-1

## The model from four GPUs
The setup cell downloaded the episode's: 30 minutes on four RTX 4090s. Bits per character
against the laptop's BPE model, and a story from each.

In [ ]:
!python bits.py checkpoints/pretrain-rope-bpe/model.pt checkpoints/pretrain-ddp/model.pt
!python sample.py checkpoints/pretrain-rope-bpe/model.pt -n 1 --tokens 160 --seed 6
!python sample.py checkpoints/pretrain-ddp/model.pt -n 1 --tokens 160 --seed 6

## Saved for transformers

In [ ]:
!python hub.py checkpoints/pretrain-ddp/model.pt hf-ddp 2>/dev/null

Next: [episode 13, Flash attention from scratch: PyTorch, Triton](https://colab.research.google.com/github/nenuadrian/mvGPT/blob/main/notebooks/ep013-flash-attention.ipynb)